In [ ]:
# default_exp data.torch

# PyTorch data loaders and transforms

> PyTorch DataLoaders, DataSet and transforms

In [ ]:
#hide
from nbdev.showdoc import *

In [ ]:
#|export
#nbdev_comment from __future__ import annotations
import numpy as np

from fastcore.test import *

from mirzai.data.loading import load_kssl
from mirzai.data.selection import (select_y, select_tax_order, select_X)
from mirzai.data.transform import (log_transform_y, SNV)

from sklearn.model_selection import train_test_split

from fastcore.transform import compose

import torch
from torch import nn
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset

## Loaders & datasets

In [ ]:
#|export
class SpectralDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        X = self.X[None, idx, :]
        y = self.y[None, idx]
        if self.transform:
            X = self.transform(X)
        return X.astype(np.float32), y.astype(np.float32)

In [ ]:
#|export
class DataLoaders():
    def __init__(self, data, transform=None, batch_size=32):
        """
        Convert numpy error to Pytorch data loaders (generators)
        Args:
            data: tuple as ((X_train, y_train), (X_test, y_test))
            transform: callable class (__class__)

        Returns:
            (training_generator, validation_generator)
        """
        self.data = data
        self.batch_size = batch_size
        self.transform = transform if transform else Noop()

    def loaders(self):
        return (DataLoader(SpectralDataset(X, y, transform=self.transform), batch_size=self.batch_size)
                for X, y in self.data)

## Transforms

In [ ]:
#|export
class SNV_transform():
    def __init__(self):
        None
    def __call__(self, spectrum):
        return SNV().fit_transform(spectrum)

In [ ]:
#|export
class Noop():
    def __init__(self):
        None
    def __call__(self, X):
        return X

## Example of use

### Load and preprocess data

In [ ]:
src_dir = './files'
fnames = ['spectra-features-smp.npy', 'spectra-wavenumbers-smp.npy', 
          'depth-order-smp.npy', 'target-smp.npy', 
          'tax-order-lu-smp.pkl', 'spectra-id-smp.npy']

X, X_names, depth_order, y, tax_lookup, X_id = load_kssl(src_dir, fnames=fnames)
transforms = [select_y, select_tax_order, select_X, log_transform_y]

data = X, y, X_id, depth_order
X, y, X_id, depth_order = compose(*transforms)(data)

### Train/test split

In [ ]:
data = train_test_split(X, y, depth_order, test_size=0.1, random_state=42)
X_train, X_test, y_train, y_test, depth_order_train, depth_order_test = data

### Create the generators

In [ ]:
dls = DataLoaders(((X_train, y_train), (X_test, y_test)), 
                  transform=SNV_transform())
training_generator, validation_generator = dls.loaders()

### Iterate over data (features, targets) mini batches

In [ ]:
for features, target in training_generator:
    print(f'Batch of features (spectra): {features.shape}')
    print(f'Batch of targets: {target.shape}')

Batch of features (spectra): torch.Size([32, 1, 1764])
Batch of targets: torch.Size([32, 1])
Batch of features (spectra): torch.Size([32, 1, 1764])
Batch of targets: torch.Size([32, 1])
Batch of features (spectra): torch.Size([7, 1, 1764])
Batch of targets: torch.Size([7, 1])
